In [ ]:
import random
import os
import json

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel

## Dataset

In [ ]:
class RerankDataset(Dataset):
    """
    Each sample:
        - query: str
        - chunk: str (positive passage)
    """

    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            "query": item["query"],
            "positive": item["chunk"]
        }


## Cross-Encoder Model
We build a cross-encoder model based on an encoder-only Transformer.

A query and passage are concatenated and jointly encoded as a single input sequence.

A relevance score is then computed by applying mean pooling to the encoder outputs, followed by an MLP.

In [ ]:
class E5CrossEncoder(nn.Module):
    """
    Cross-encoder built on top of a transformer encoder.

    The model encodes a concatenated input of:
        "query: ... passage: ..."

    and produces a single relevance score per input pair.

    Architecture:
        Transformer encoder → mean pooling → MLP scoring head
    """

    def __init__(self, encoder):
        super().__init__()

        self.encoder = encoder
        hidden_size = encoder.config.hidden_size

        # self.scoring = nn.Linear(hidden_size, 1)
        self.scoring = nn.Sequential(nn.Linear(hidden_size, hidden_size // 2),
                                     nn.ReLU(),
                                     nn.Dropout(0.1),
                                     nn.Linear(hidden_size // 2, hidden_size // 4),
                                     nn.ReLU(),
                                     nn.Dropout(0.1),
                                     nn.Linear(hidden_size // 4, 1))

    def forward(self, input_ids, attention_mask):

        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # token_embeddings: [B, T, H]
        token_embeddings = outputs.last_hidden_state

        # expanded_mask: [B, T, 1]
        expanded_mask = attention_mask.unsqueeze(-1).to(dtype=token_embeddings.dtype)

        # numerator: [B, H]
        numerator = (token_embeddings * expanded_mask).sum(dim=1)

        # denominator: [B, 1]
        denominator = expanded_mask.sum(dim=1).clamp(min=1e-9)

        # pooled: [B, H]
        pooled = numerator / denominator

        # scores: [B]
        scores = self.scoring(pooled).squeeze(-1)

        return scores


## Reranker wrapper for training and inference
The training objective corresponds to Information Noise-Contrastive Estimation (InfoNCE).

The training scheme uses in-batch negatives, where each query is paired with one positive example in the batch and all other examples act as negatives, following Multiple Negatives Ranking Loss (MNRL).

At the implementation level, the objective is computed as a CrossEntropy loss over pairwise similarity scores.


In [ ]:
class E5Reranker:
    """
    Cross-encoder reranker based on an E5 encoder where only the last two transformer layers and the scoring head are fine-tuned.

    Training uses in-batch negatives (all other positives in the batch act as negatives).
    The objective is implemented as a CrossEntropy loss over pairwise similarity scores, which is equivalent to the Multiple Negatives Ranking Loss (MNRL), a special case of InfoNCE.
    """

    def __init__(self, model_name="intfloat/multilingual-e5-base", device=None):

        self.model_name = model_name
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        encoder = AutoModel.from_pretrained(model_name)
        self.model = E5CrossEncoder(encoder)

        # Device setup
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)

        # Transformer encoder layers
        layers = encoder.encoder.layer

        # Freeze full encoder
        for p in self.model.encoder.parameters():
            p.requires_grad = False

        # Unfreeze last 2 encoder layers for partial fine-tuning
        # Adapt high-level semantic representations while keeping most weights frozen
        for layer in layers[-2:]:
            for p in layer.parameters():
                p.requires_grad = True

        # Collect encoder parameters that will be optimized (only last two layers)
        encoder_trainable_params = []
        for layer in layers[-2:]:
            encoder_trainable_params.extend(layer.parameters())

        # GradScaler for mixed precision training stability
        # Uses dynamic loss scaling to prevent underflow/overflow in FP16 gradients
        # and ensures safe optimizer steps during backpropagation
        self.scaler = torch.cuda.amp.GradScaler()

        # Optimizer setup.
        # The scoring head learns faster because it is initialized from scratch,
        # while the encoder is pretrained and only lightly fine-tuned.
        self.optimizer = torch.optim.AdamW([
            {"params": self.model.scoring.parameters(), "lr": 2e-5},
            {"params": encoder_trainable_params, "lr": 1e-5}
        ])

    def build_inputs(self, queries, passages):
        """
        Build cross-encoder inputs from flattened lists of queries and passages.

        The inputs correspond to all query–passage pairs produced in the training step
        (i.e., after expanding a batch of size B into B × B combinations).

        Both inputs are provided as parallel flattened lists:
        - queries[i] is paired with passages[i]

        The model is a cross-encoder, so each pair is processed jointly in a single sequence,
        enabling full token-level interaction between query and passage.

        Format:
            "query: {query} passage: {passage}"

        Returns:
            Tokenized batch ready for the transformer model.
        """


        texts = [
            f"query: {q} passage: {p}"
            for q, p in zip(queries, passages)
        ]

        return self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        )


    def forward_scores(self, queries, positives):
        """
        Compute a B×B similarity matrix using in-batch negatives.

        This function constructs all pairwise combinations between queries and
        positives in the batch, producing a dense similarity matrix used for
        both training and evaluation.

        Diagonal elements correspond to correct query–positive pairs, while
        off-diagonal elements act as implicit negatives under the in-batch
        ranking setup commonly used in contrastive retrieval models.

        The function is used consistently across training, validation, and test
        to ensure comparable retrieval metrics (e.g., Recall@k and MRR).

        The cross-encoder forward pass is executed in mixed precision (AMP
        autocast) for efficiency.

        Args:
            queries (List[str]): Batch of queries [q1, ..., qB].
            positives (List[str]): Corresponding positive passages [p1, ..., pB].

        Returns:
            torch.Tensor: B×B similarity matrix where rows correspond to queries
            and columns correspond to candidate passages.
        """

        # Build a BxB similarity matrix from B (query, positive) pairs
        B = len(queries)
        queries_expanded = []
        positives_expanded = []
        for q in queries:
            queries_expanded.extend([q] * B)
            positives_expanded.extend(positives)

        # Build inputs (tokenize query-passage pairs)
        inputs = self.build_inputs(queries_expanded, positives_expanded)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        # Mixed precision forward pass
        with torch.cuda.amp.autocast():
            scores = self.model(**inputs)

        # Reshape scores so each row corresponds to one query and each column to one passage
        return scores.view(B, B)


    @staticmethod
    def compute_metrics(scores, labels, k_list=(1, 3)):
        """
        Compute retrieval metrics for a B x B similarity matrix.

        Args:
            scores (Tensor): [B, B] similarity matrix
            labels (Tensor): [B] index of positive passage per query
            k_list (tuple): Recall@k values to compute

        Returns:
            dict: recall@k and mrr
        """

        # Rank indices (highest score first)
        _, ranked = scores.topk(k=scores.size(1), dim=1, largest=True)

        labels_expanded = labels.unsqueeze(1)

        metrics = {}

        # Recall@k
        for k in k_list:
            topk = ranked[:, :k]
            hits = (topk == labels_expanded).any(dim=1)
            metrics[f"recall@{k}"] = hits.float().mean().item()

        # MRR
        ranks = (ranked == labels_expanded).nonzero(as_tuple=True)

        # ranks[0] = batch index
        # ranks[1] = position in ranking
        positive_ranks = ranks[1] + 1

        mrr = (1.0 / positive_ranks.float()).mean().item()
        metrics["mrr"] = mrr

        return metrics


    def train_step(self, queries, positives):
        """
        Perform one training step using in-batch negatives (InfoNCE / MNRL).

        Given a batch of B (query, positive) pairs, the model builds a B×B similarity
        matrix where each query is compared against all positives in the batch.
        Correct matches lie on the diagonal, while off-diagonal elements act as
        implicit negatives.

        The model is trained with row-wise cross-entropy, forcing each query to
        select its correct passage among B candidates. This is equivalent to
        InfoNCE / Multiple Negatives Ranking Loss.

        Steps:
        1. Compute B×B similarity scores via `forward_scores`.
        2. Define diagonal labels as ground truth.
        3. Apply cross-entropy loss.
        4. Backpropagate with mixed precision scaling.
        5. Compute retrieval metrics (Recall@k, MRR) for monitoring.

        Returns:
            dict with training loss and retrieval metrics.
        """

        # Reset accumulated gradients
        self.optimizer.zero_grad()

        scores = self.forward_scores(queries, positives)

        # Create labels indicating that each query matches the passage at the same index
        labels = torch.arange(len(queries), device=self.device)

        # Cross-entropy over the B×B similarity matrix (contrastive learning).
        loss = F.cross_entropy(scores, labels)

        # Backpropagation and parameter update
        # Uses dynamic loss scaling for numerical stability in mixed precision
        self.scaler.scale(loss).backward()
        self.scaler.step(self.optimizer)
        self.scaler.update()

        # Compute retrieval metrics (Recall@k and MRR) from the similarity matrix
        # and extract values for logging/monitoring
        metrics = E5Reranker.compute_metrics(scores, labels)

        # Return training loss and computed retrieval metrics as a unified dictionary
        return {
            "loss": loss.item(),
            **metrics
        }

    @torch.no_grad()
    def evaluate(self, dataloader):
        """
        Evaluate the model on a dataset without updating its parameters.

        For each batch, the method computes the B×B similarity matrix, the
        cross-entropy loss, and the retrieval metrics (Recall@1, Recall@3, MRR).
        The returned values are the averages across all batches.

        Args:
            dataloader: evaluation dataloader.

        Returns:
            dict containing the mean loss and retrieval metrics.
        """

        losses = []
        r1s = []
        r3s = []
        mrrs = []

        for batch in dataloader:

            scores = self.forward_scores(
                batch["query"],
                batch["positive"]
            )

            labels = torch.arange(len(batch["query"]), device=self.device)

            loss = F.cross_entropy(scores, labels)
            metrics = E5Reranker.compute_metrics(scores, labels)

            losses.append(loss.item())
            r1s.append(metrics["recall@1"])
            r3s.append(metrics["recall@3"])
            mrrs.append(metrics["mrr"])

        return {
            "loss": sum(losses) / len(losses),
            "recall@1": sum(r1s) / len(r1s),
            "recall@3": sum(r3s) / len(r3s),
            "mrr": sum(mrrs) / len(mrrs)
        }


    def fit(self, train_loader, val_loader=None, epochs=10, log_every=20, patience=2):

        """
        Training loop for a cross-encoder retrieval model.

        The model learns using in-batch negatives: each query is trained to
        pick its correct passage from the rest of the batch.

        We track standard retrieval metrics during training and validation:
            - Recall@1
            - Recall@3
            - MRR (main metric for model selection)
            - Cross-entropy loss

        How training works:
            - Trains for a fixed number of epochs
            - Logs batch and epoch-level metrics to monitor progress
            - Uses a sliding window (log_every) to smooth training signals

        Validation:
            - Runs at the end of each epoch (if provided)
            - Used to check generalization and guide model selection

        Early stopping:
            - Stops training if MRR does not improve for several epochs
            - Keeps the best model seen during training (based on MRR + loss tie-break)

        Args:
            train_loader: training data
            val_loader: optional validation data
            epochs: number of training epochs
            log_every: how often to average and log metrics
            patience: how many epochs to wait before stopping if no improvement
        """

        self.log_every = log_every
        self.val_batches = []

        self.loss_history = []
        self.recall1_history = []
        self.recall3_history = []
        self.mrr_history = []

        self.train_loss_epoch_history = []
        self.train_recall1_epoch_history = []
        self.train_recall3_epoch_history = []
        self.train_mrr_epoch_history = []

        self.val_loss_history = []
        self.val_recall1_history = []
        self.val_recall3_history = []
        self.val_mrr_history = []

        best_mrr = -float("inf")
        best_loss = float("inf")
        best_state = None
        no_improve_epochs = 0

        for epoch in range(epochs):

            running = {"loss": 0.0, "r1": 0.0, "r3": 0.0, "mrr": 0.0}
            epoch_metrics = []

            # Train phase
            # Set the E5CrossEncoder model in training mode (activate dropout)
            self.model.train()
            for step, batch in enumerate(train_loader):

                out = self.train_step(
                    batch["query"],
                    batch["positive"]
                )

                running["loss"] += out["loss"]
                running["r1"] += out["recall@1"]
                running["r3"] += out["recall@3"]
                running["mrr"] += out["mrr"]

                # Intermediate logging (optional)
                if step % 50 == 0:
                    print(
                        f"Epoch {epoch+1} | Step {step} | "
                        f"Loss {out['loss']:.4f} | "
                        f"R@1 {out['recall@1']:.3f} | "
                        f"R@3 {out['recall@3']:.3f} | "
                        f"MRR {out['mrr']:.3f}"
                    )


                if (step + 1) % log_every == 0:
                    avg = {k: v / log_every for k, v in running.items()}

                    self.loss_history.append(avg["loss"])
                    self.recall1_history.append(avg["r1"])
                    self.recall3_history.append(avg["r3"])
                    self.mrr_history.append(avg["mrr"])

                    epoch_metrics.append(avg)

                    running = {k: 0.0 for k in running}

            # Epoch summary
            if len(epoch_metrics) > 0:

                keys = ["loss", "r1", "r3", "mrr"]
                mean = {k: sum(m[k] for m in epoch_metrics) / len(epoch_metrics) for k in keys}

                self.train_loss_epoch_history.append(mean["loss"])
                self.train_recall1_epoch_history.append(mean["r1"])
                self.train_recall3_epoch_history.append(mean["r3"])
                self.train_mrr_epoch_history.append(mean["mrr"])

                print(
                    f"\nEpoch {epoch+1} | "
                    f"Loss {mean['loss']:.4f} | "
                    f"R@1 {mean['r1']:.3f} | "
                    f"R@3 {mean['r3']:.3f} | "
                    f"MRR {mean['mrr']:.3f}"
                )

            # Validation phase
            if val_loader is not None:

                self.model.eval()
                val_metrics = self.evaluate(val_loader)

                self.val_batches.append((epoch + 1) * len(train_loader))

                self.val_loss_history.append(val_metrics["loss"])
                self.val_recall1_history.append(val_metrics["recall@1"])
                self.val_recall3_history.append(val_metrics["recall@3"])
                self.val_mrr_history.append(val_metrics["mrr"])

                print(
                    f"Val Loss {val_metrics['loss']:.4f} | "
                    f"R@1 {val_metrics['recall@1']:.3f} | "
                    f"R@3 {val_metrics['recall@3']:.3f} | "
                    f"MRR {val_metrics['mrr']:.3f}"
                )

                # Early Stopping (MRR)
                if (
                     val_metrics["mrr"] > best_mrr + 1e-4 or
                     (abs(val_metrics["mrr"] - best_mrr) <= 1e-4
                     and val_metrics["loss"] < best_loss)
                   ):
                        best_mrr = val_metrics["mrr"]
                        best_loss = val_metrics["loss"]
                        no_improve_epochs = 0

                        best_state = {
                            k: v.cpu().clone()
                            for k, v in self.model.state_dict().items()
                        }
                else:
                    no_improve_epochs += 1

                if no_improve_epochs >= patience:
                    print(f"\nEarly stopping at epoch {epoch+1}")
                    break

        # Restore best model
        if best_state is not None:
            self.model.load_state_dict(best_state)

            if val_loader is not None:
                final_metrics = self.evaluate(val_loader)

                print(
                       f"Restored model -> Loss {final_metrics['loss']:.4f} | "
                       f"R@1 {final_metrics['recall@1']:.3f} | "
                       f"R@3 {final_metrics['recall@3']:.3f} | "
                       f"MRR {final_metrics['mrr']:.3f}"
                )

    @torch.no_grad()
    def rerank(self, query, passages, top_k=None):
        """
        Rank the candidate passages according to their relevance to the query.

        Args:
            query (str): User query.
            passages (List[str]): Candidate passages to rerank.
            top_k (int, optional): Number of highest-ranked passages to return.
                If None, all passages are returned.

        Returns:
            List[Tuple[str, float]]: Ranked (passage, score) pairs sorted in
            descending order of relevance.
        """

        self.model.eval()

        queries = [query] * len(passages)

        inputs = self.build_inputs(queries, passages)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        scores = self.model(**inputs).squeeze(-1).cpu()

        ranked_idx = torch.argsort(scores, descending=True)

        if top_k is not None:
            ranked_idx = ranked_idx[:top_k]

        return [
            (passages[i], scores[i].item())
            for i in ranked_idx.tolist()
        ]


    def save(self, path):
        """
        Save the reranker model, tokenizer and configuration to disk.
        """
        os.makedirs(path, exist_ok=True)
        torch.save(self.model.state_dict(), os.path.join(path, "model.pt"))
        self.tokenizer.save_pretrained(path)
        with open(os.path.join(path, "config.txt"), "w") as f:
            f.write(self.model_name)


    @staticmethod
    def load(path, device=None):
        """
        Load a previously saved reranker from disk.
        """

        device = device or ("cuda" if torch.cuda.is_available() else "cpu")

        model_name = open(f"{path}/config.txt").read().strip()

        tokenizer = AutoTokenizer.from_pretrained(path)

        encoder = AutoModel.from_pretrained(model_name)
        model = E5CrossEncoder(encoder)

        state_dict = torch.load(f"{path}/model.pt", map_location=device)
        model.load_state_dict(state_dict)

        model.to(device)
        model.eval()

        reranker = E5Reranker.__new__(E5Reranker)
        reranker.model = model
        reranker.tokenizer = tokenizer
        reranker.model_name = model_name
        reranker.device = device
        reranker.optimizer = None

        return reranker


## Training and Inference

In [ ]:
from google.colab import drive
import random
from sklearn.model_selection import train_test_split

drive.mount("/content/drive", force_remount=False)

DATASET_DIR = "/content/drive/MyDrive/RAG_UPC_Final_project/construction_related_documents"
dataset_file_name = "dataset_queries_paragraphs_1782985692.json" # @param {type:"string"}
dataset_file_path = f"{DATASET_DIR}/{dataset_file_name}"

BATCH_SIZE = 12
EPOCHS = 10
PATIENCE = 2
MODEL_NAME = "intfloat/multilingual-e5-base"
MODEL_DIR = DATASET_DIR

# Load data
with open(dataset_file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# Splits data into training data (80%), validation data (10%),
# and testing data (10%)
random.seed(42)
random.shuffle(data)

train_data, temp_data = train_test_split(
    data,
    test_size=0.2,
    random_state=42
)

val_data, test_data = train_test_split(
    temp_data,
    test_size=0.5,
    random_state=42
)

# Shows dataset sizes
header = "Dataset sizes"
msg1 = f"- Training dataset: {len(train_data)} samples"
msg2 = f"- Validation dataset: {len(val_data)} samples"
msg3 = f"- Testing dataset: {len(test_data)} samples"
print("=" * len(header))
print(header)
print("=" * len(header))
print(f"- Training dataset: {len(train_data)} samples")
print(f"- Validation dataset: {len(val_data)} samples")
print(f"- Testing dataset: {len(test_data)} samples")
print("-" * max(len(msg1), len(msg2), len(msg3)))

# Build Dataset objects
train_dataset = RerankDataset(train_data)
val_dataset = RerankDataset(val_data)
test_dataset = RerankDataset(test_data)

# Build DataLoader objects
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Builds E5Reranker object
reranker = E5Reranker(model_name=MODEL_NAME)

# Trains the reranker
print("Starting training...\n")
reranker.fit(
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=EPOCHS,
    patience=PATIENCE
)

# Evaluation on the test set (Inference)
test_metrics = reranker.evaluate(test_loader)
print(
    f"\nTEST -> "
    f"Loss {test_metrics['loss']:.4f} | "
    f"R@1 {test_metrics['recall@1']:.3f} | "
    f"R@3 {test_metrics['recall@3']:.3f} | "
    f"MRR {test_metrics['mrr']:.3f}"
)

# Save model
reranker.save(MODEL_DIR)
print(f"\nModel saved to: {MODEL_DIR}")

In [ ]:
import matplotlib.pyplot as plt

# Data summary
print("=" * 60)
print("DATASET SUMMARY")
print("=" * 60)
print(f"Epochs: {EPOCHS}")
print(f"Early stopping: enabled (patience = {PATIENCE})")
print(f"Batch size: {BATCH_SIZE}")
print(f"Train samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")
print(f"Test samples: {len(test_data)}")
print("=" * 60)

test = test_metrics

# X axis
x_train_batch = [
    (i + 1) * reranker.log_every
    for i in range(len(reranker.loss_history))
]

x_epoch = list(range(1, len(reranker.train_loss_epoch_history) + 1))

# Figure 1: train (batch-level)
fig1, axes1 = plt.subplots(2, 2, figsize=(12, 8))

# Loss
axes1[0,0].plot(x_train_batch, reranker.loss_history, color="blue")
axes1[0,0].set_title("Train Loss (batch)")
axes1[0,0].grid(True)

# Recall@1
axes1[0,1].plot(x_train_batch, reranker.recall1_history, color="blue")
axes1[0,1].set_title("Train Recall@1 (batch)")
axes1[0,1].grid(True)

# Recall@3
axes1[1,0].plot(x_train_batch, reranker.recall3_history, color="blue")
axes1[1,0].set_title("Train Recall@3 (batch)")
axes1[1,0].grid(True)

# MRR
axes1[1,1].plot(x_train_batch, reranker.mrr_history, color="blue")
axes1[1,1].set_title("Train MRR (batch)")
axes1[1,1].grid(True)


# Figure 2: train vs validation (epoch-level)=
fig2, axes2 = plt.subplots(2, 2, figsize=(12, 8))

# Loss
axes2[0,0].plot(x_epoch, reranker.train_loss_epoch_history,
                color="blue", marker="o", label="Train")
axes2[0,0].plot(x_epoch, reranker.val_loss_history,
                color="orange", marker="o", label="Validation")
axes2[0,0].set_title("Loss (epoch)")
axes2[0,0].set_xticks(x_epoch)
axes2[0,0].grid(True)
axes2[0,0].legend()

# Recall@1
axes2[0,1].plot(x_epoch, reranker.train_recall1_epoch_history,
                color="blue", marker="o", label="Train")
axes2[0,1].plot(x_epoch, reranker.val_recall1_history,
                color="orange", marker="o", label="Validation")
axes2[0,1].set_title("Recall@1 (epoch)")
axes2[0,1].set_xticks(x_epoch)
axes2[0,1].grid(True)
axes2[0,1].legend()

# Recall@3
axes2[1,0].plot(x_epoch, reranker.train_recall3_epoch_history,
                color="blue", marker="o", label="Train")
axes2[1,0].plot(x_epoch, reranker.val_recall3_history,
                color="orange", marker="o", label="Validation")
axes2[1,0].set_title("Recall@3 (epoch)")
axes2[1,0].set_xticks(x_epoch)
axes2[1,0].grid(True)
axes2[1,0].legend()


# MRR
axes2[1,1].plot(x_epoch, reranker.train_mrr_epoch_history,
                color="blue", marker="o", label="Train")
axes2[1,1].plot(x_epoch, reranker.val_mrr_history,
                color="orange", marker="o", label="Validation")
axes2[1,1].set_title("MRR (epoch)")
axes2[1,1].set_xticks(x_epoch)
axes2[1,1].grid(True)
axes2[1,1].legend()

plt.tight_layout()
plt.show()


# Test results (numeric only)
print("\n" + "=" * 60)
print("TEST RESULTS (FINAL MODEL)")
print("=" * 60)
print(f"Loss:     {test['loss']:.4f}")
print(f"Recall@1: {test['recall@1']:.3f}")
print(f"Recall@3: {test['recall@3']:.3f}")
print(f"MRR:      {test['mrr']:.3f}")
print("=" * 60)